# Kenzan: run the VLM on held-out v3 (Colab)

Runtime → Change runtime type → **T4 GPU** (free) is enough for the default (Qwen3-VL-8B-Instruct, 4-bit).
This notebook only produces raw model responses (`vlm_cache/`). Scoring is done offline with `stress/run_vlm_compare.py`, which never calls a model.

Expected time on a T4: model download ~5 min, then a few minutes per page (8 pages).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone -q https://github.com/vedantjhunjhunwal/Kenzan.git
%cd Kenzan
!pip install -q -r requirements-vlm.txt

Regenerate the 8 degraded pages and check their image hashes against the frozen list. If any line says MISMATCH, upload `heldout_v3_pdfs.zip` instead and run the unzip line below.

In [ ]:
!python stress/make_pdfs.py stress/specs/heldout_v3.json stress/out_heldout_v3/pdfs
# only if MISMATCH: upload heldout_v3_pdfs.zip with the file browser, then:
# !unzip -o ../heldout_v3_pdfs.zip -d stress/out_heldout_v3/pdfs

In [ ]:
!python scripts/vlm_run.py --pdfs stress/out_heldout_v3/pdfs --cache vlm_cache --quant 4bit
# with an L4/A100 (>= 24 GB) you can also run --quant none; it is cached under a different key

In [ ]:
!zip -qr vlm_cache.zip vlm_cache
from google.colab import files
files.download('vlm_cache.zip')